Recipe & Meal Plan Dataset Generator
Generate synthetic recipe and meal-plan datasets via a local LLM. Describe what you want (e.g. "30 quick weeknight dinners with prep time and ingredients") and get back structured data.

Features

Output as CSV or Markdown table (you choose in the chat).
Runs on a single GPU; 4-bit quantization (BitsAndBytes) keeps memory low so it fits on Colab A100 free tier or similar.
Built with Hugging Face Transformers; responses stream token-by-token in the UI.

In [ ]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate gradio

In [ ]:
# Model and generation settings
# MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MODEL_ID="Qwen/Qwen2.5-7B-Instruct"
DEFAULT_N_ROWS = 50
MAX_NEW_TOKENS = 8192

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextIteratorStreamer
from huggingface_hub import login
# from google.colab import userdata only required for colab
import os # for local machine disable in colab
from dotenv import load_dotenv # for local machine disable in colab
import torch
import gradio as gr
from threading import Thread

In [ ]:
#huggingface setup
# hf_token = userdata.get("HF_TOKEN") for colab
load_dotenv() # for local machine disable in colab
hf_token = os.getenv("HF_TOKEN") # for local machine disable in colab
login(hf_token, add_to_git_credential=True)

In [ ]:
def load_model_and_tokenizer(model_id):
    """Load model with 4-bit quantization for lower VRAM (e.g. Colab A100)."""
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant_config)
    return tokenizer, model

In [ ]:
print("Loading model and tokenizer...")
tokenizer, model = load_model_and_tokenizer(MODEL_ID)
print("Ready.")

In [ ]:
# System prompt: recipe & meal-plan dataset only
RECIPE_SYSTEM_PROMPT = """You are Marlow, a recipe and meal-plan data curator. Your only job is to generate synthetic recipe or meal-plan datasets.

RULES:
- Only fulfill requests for recipe, meal-plan, or food-related tabular data. Politely decline anything else (e.g. code, other topics).
- When the user asks for a dataset, output ONLY the data table—no intro, no "here is...", no explanations.
- Default size: 50 rows unless the user specifies another number.
- Ask the user once whether they want the result in CSV or Markdown table format if they have not already said.
- Include rich, realistic columns (e.g. recipe_name, cuisine, prep_minutes, cook_minutes, ingredients, dietary_tags, servings).
- Keep conversation brief; focus on producing the requested dataset."""

def build_messages(history, user_prompt):
    messages = [{"role": "system", "content": RECIPE_SYSTEM_PROMPT}]
    messages += [{"role": h["role"], "content": h["content"]} for h in (history or [])]
    messages.append({"role": "user", "content": user_prompt})
    return messages

def create_inputs(tokenizer, history, prompt):
    messages = build_messages(history, prompt)
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

In [ ]:
def chat(prompt, history):
    inputs = create_inputs(tokenizer, history, prompt)
    if not isinstance(inputs, dict):
        inputs = {"input_ids": inputs}
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
        decode_kwargs={"skip_special_tokens": True},
    )
    thread = Thread(
        target=model.generate,
        kwargs={
            **inputs,
            "max_new_tokens": MAX_NEW_TOKENS,
            "streamer": streamer,
        },
    )
    thread.start()
    full_response = ""
    for chunk in streamer:
        full_response += chunk.replace("<|eot_id|>", "")
        yield full_response

In [ ]:
# Gradio UI: chat to request recipe/meal-plan datasets
with gr.Blocks(title="Recipe & Meal Plan Dataset Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("Ask **Marlow** for a recipe or meal-plan dataset (e.g. *20 vegetarian lunches in CSV*). Say how many rows and CSV vs Markdown.")
    chatbot = gr.Chatbot(height=420, type="messages", label="Chat")
    msg = gr.Textbox(placeholder="e.g. 30 quick weeknight dinners, markdown table with ingredients and cook time", label="Request", lines=2)
    submit = gr.Button("Generate", variant="primary")

    def respond(message, history):
        history = list(history or [])
        user_msg = {"role": "user", "content": message}
        for full_reply in chat(message, history):
            yield history + [user_msg, {"role": "assistant", "content": full_reply}]

    submit.click(respond, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", None, msg)
    msg.submit(respond, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", None, msg)

demo.queue()
demo.launch()